<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 08. Bahdanau Additive Attention — Neural Machine Translation by Jointly Learning to Align and Translate

- **원 논문:** [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)
- **저자 / 발표:** Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio · ICLR 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** encoder-decoder, softmax, masking, batched matrix multiplication

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

decoder query와 모든 encoder annotation 사이 additive energy를 계산하고, padding을 softmax 전에 가린 뒤 정렬 확률의 가중합으로 context를 만든다. 작은 단조 정렬 과제에서 attention weight가 올바른 위치를 찾는지 측정한다.

**원 논문과 다른 것**

원 논문은 양방향 RNN encoder와 gated decoder를 WMT'14 영어-프랑스어 번역에 학습해 BLEU와 alignment를 평가했다. 여기서는 이미 만들어진 6차원 annotation과 5개 query만 사용하며 번역기 전체나 BLEU를 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1, 식 (6): context vector는 annotation의 가중합 | `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)` | context shape와 attention 행 합 1을 assert |
| §3.1, 식 (7): energy를 softmax로 정규화해 α_ij 계산 | `AdditiveAttention.forward`의 masked energy와 `softmax(dim=-1)` | padding 열의 확률이 정확히 0인지 검사 |
| §3.1, 식 (8) 및 Appendix A.1.2: feed-forward alignment model a(s,h) | `AdditiveAttention.score`의 `v(tanh(W_s s + W_h h))` | energy shape 및 모든 파라미터 gradient 확인 |
| Figure 3: target 위치와 source annotation 사이 soft alignment | 합성 monotonic alignment 정확도와 attention heatmap | argmax alignment accuracy 및 padding mass로 정량 검증 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
e_{t,i}=v_a^\top\tanh(W_s s_{t-1}+W_h h_i),\quad
\alpha_{t,i}=\operatorname{softmax}_i(e_{t,i}),\quad
c_t=\sum_i\alpha_{t,i}h_i
$$

- $h_i$는 encoder state, $s_{t-1}$는 decoder state, $\alpha_{t,i}$는 alignment weight입니다.
- 하나의 고정 context 대신 decoder step마다 source의 다른 위치를 가중 합산합니다.
- energy, masked softmax, context aggregation을 분리해 alignment을 직접 검증합니다.
- score와 weight는 $[B,T,S]$, context는 $[B,T,H]$이며 padding 위치 가중치는 0이어야 합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1, 식 (6): context vector는 annotation의 가중합
- **노트북 구현:** `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)`
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** context shape와 attention 행 합 1을 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 양방향 RNN encoder와 gated decoder를 WMT'14 영어-프랑스어 번역에 학습해 BLEU와 alignment를 평가했다. 여기서는 이미 만들어진 6차원 annotation과 5개 query만 사용하며 번역기 전체나 BLEU를 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

하나의 고정 길이 벡터 대신 decoder step $i$마다
$e_{ij}=v_a^\top\tanh(W_a s_{i-1}+U_a h_j)$를 계산한다. 이를 source 축으로
softmax한 $\alpha_{ij}$가 soft alignment이고,
$c_i=\sum_j\alpha_{ij}h_j$가 현재 step의 context다(식 (6)--(8)).

논문에는 padding mask가 수식으로 명시되지 않지만, minibatch 구현에서는 가짜
annotation이 확률 질량을 받지 않도록 energy를 softmax **전에** 가려야 한다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 808
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())

# 다섯 decoder step, 여섯 encoder slot(마지막은 padding)
STEPS, SOURCE_SLOTS, STATE_DIM = 5, 6, 6
annotations = torch.eye(SOURCE_SLOTS).unsqueeze(0).expand(STEPS, -1, -1).clone()
decoder_queries = torch.eye(STATE_DIM)[:STEPS]
source_mask = torch.ones((STEPS, SOURCE_SLOTS), dtype=torch.bool)
source_mask[:, -1] = False
alignment_targets = torch.arange(STEPS)
annotations, decoder_queries, source_mask, alignment_targets = ACCELERATOR.move(
    annotations, decoder_queries, source_mask, alignment_targets
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, 식 (6): context vector는 annotation의 가중합 / §3.1, 식 (7): energy를 softmax로 정규화해 α_ij 계산 / §3.1, 식 (8) 및 Appendix A.1.2: feed-forward alignment model a(s,h)
- **이 셀의 책임:** `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)` / `AdditiveAttention.forward`의 masked energy와 `softmax(dim=-1)` / `AdditiveAttention.score`의 `v(tanh(W_s s + W_h h))`
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** context shape와 attention 행 합 1을 assert / padding 열의 확률이 정확히 0인지 검사 / energy shape 및 모든 파라미터 gradient 확인. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, annotation_dim, attention_dim):
        super().__init__()
        self.query_projection = nn.Linear(query_dim, attention_dim, bias=False)
        self.annotation_projection = nn.Linear(
            annotation_dim,
            attention_dim,
            bias=False,
        )
        self.energy_projection = nn.Linear(attention_dim, 1, bias=False)

    def score(self, query, encoder_annotations):
        if query.ndim != 2 or encoder_annotations.ndim != 3:
            raise ValueError("expected query [B,D] and annotations [B,S,D]")
        query_term = self.query_projection(query).unsqueeze(1)
        annotation_term = self.annotation_projection(encoder_annotations)
        joint_state = torch.tanh(query_term + annotation_term)
        return self.energy_projection(joint_state).squeeze(-1)

    def normalize(self, energy, valid_mask):
        if energy.shape != valid_mask.shape:
            raise ValueError("energy and valid_mask shapes must match")
        if not valid_mask.any(dim=1).all():
            raise ValueError("every sequence needs at least one valid annotation")
        masked_energy = energy.masked_fill(
            ~valid_mask,
            torch.finfo(energy.dtype).min,
        )
        alpha = torch.softmax(masked_energy, dim=-1)
        return alpha, masked_energy

    def compose_context(self, alpha, encoder_annotations):
        weighted = torch.bmm(alpha.unsqueeze(1), encoder_annotations)
        return weighted.squeeze(1)

    def forward(self, query, encoder_annotations, valid_mask):
        energy = self.score(query, encoder_annotations)
        alpha, masked_energy = self.normalize(energy, valid_mask)
        context = self.compose_context(alpha, encoder_annotations)
        return context, alpha, masked_energy


torch.manual_seed(SEED)
attention = AdditiveAttention(STATE_DIM, STATE_DIM, attention_dim=12).to(DEVICE)
context, alpha, energy = attention(decoder_queries, annotations, source_mask)
assert context.shape == (STEPS, STATE_DIM)
assert alpha.shape == (STEPS, SOURCE_SLOTS)
assert torch.allclose(alpha.sum(dim=1), alpha.new_ones(STEPS), atol=1e-6)
assert torch.all(alpha[:, -1] == 0)

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3.1, 식 (6): context vector는 annotation의 가중합 / §3.1, 식 (7): energy를 softmax로 정규화해 α_ij 계산 / §3.1, 식 (8) 및 Appendix A.1.2: feed-forward alignment model a(s,h) / Figure 3: target 위치와 source annotation 사이 soft alignment
- **이 셀의 책임:** `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)` / `AdditiveAttention.forward`의 masked energy와 `softmax(dim=-1)` / `AdditiveAttention.score`의 `v(tanh(W_s s + W_h h))` / 합성 monotonic alignment 정확도와 attention heatmap
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** context shape와 attention 행 합 1을 assert / padding 열의 확률이 정확히 0인지 검사 / energy shape 및 모든 파라미터 gradient 확인 / argmax alignment accuracy 및 padding mass로 정량 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
optimizer = torch.optim.Adam(attention.parameters(), lr=0.05)
loss_history = []
for _ in range(140):
    optimizer.zero_grad()
    _, _, energy = attention(decoder_queries, annotations, source_mask)
    loss = F.cross_entropy(energy, alignment_targets)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

assert loss_history[-1] < 0.2 * loss_history[0]
for parameter in attention.parameters():
    assert parameter.grad is not None and torch.isfinite(parameter.grad).all()

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 3: target 위치와 source annotation 사이 soft alignment
- **이 셀의 책임:** 합성 monotonic alignment 정확도와 attention heatmap
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** argmax alignment accuracy 및 padding mass로 정량 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
with torch.inference_mode():
    learned_context, learned_alpha, _ = attention(
        decoder_queries, annotations, source_mask
    )
predicted_alignment = learned_alpha.argmax(dim=1)
alignment_accuracy = (predicted_alignment == alignment_targets).float().mean().item()
padding_mass = learned_alpha[:, -1].sum().item()
identity_context = torch.eye(STEPS, device=DEVICE)
context_error = (learned_context[:, :STEPS] - identity_context).abs().mean().item()

assert alignment_accuracy >= 0.8
assert padding_mass == 0.0
assert context_error < 0.1
print(
    f"alignment_accuracy={alignment_accuracy:.1%}, "
    f"padding_mass={padding_mass:.1e}, "
    f"context_MAE={context_error:.4f}"
)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Figure 3: target 위치와 source annotation 사이 soft alignment
- **이 셀의 책임:** 합성 monotonic alignment 정확도와 attention heatmap
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** argmax alignment accuracy 및 padding mass로 정량 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].plot(loss_history)
axes[0].set(title="alignment NLL", xlabel="step", ylabel="loss")
image = axes[1].imshow(
    learned_alpha.detach().cpu(),
    vmin=0,
    vmax=1,
    cmap="Blues",
    aspect="auto",
)
axes[1].set(
    title="soft alignment alpha(i,j)",
    xlabel="encoder position j (last = PAD)",
    ylabel="decoder step i",
    xticks=range(SOURCE_SLOTS),
    yticks=range(STEPS),
)
fig.colorbar(image, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

## 해석 체크

1. 왜 `softmax` 뒤가 아니라 앞에서 padding energy를 가려야 할까요?
2. context가 decoder step마다 달라지는 것이 고정 길이 encoder-decoder 병목을
   어떻게 완화하는지 설명하세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 양방향 RNN encoder와 gated decoder를 WMT'14 영어-프랑스어 번역에 학습해 BLEU와 alignment를 평가했다. 여기서는 이미 만들어진 6차원 annotation과 5개 query만 사용하며 번역기 전체나 BLEU를 재현하지 않는다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.